In [0]:
customer360_df = spark.table(
    "customer360.gold.customer_360"
)

In [0]:
insurance_base_df = customer360_df.select(
    "customer_id",
    "age",
    "dependents",
    "current_balance",
    "engagement_score",
    "net_worth_category"
)

display(
    insurance_base_df.summary()
)

In [0]:
from pyspark.sql.functions import when, col

insurance_df = (
    customer360_df
    .withColumn(
        "insurance_v1",
        when(
            (col("age") >= 45) &
            (col("dependents") >= 1) &
            (col("current_balance") >= 3000),
            1
        ).otherwise(0)
    )
)

In [0]:
insurance_features = [
    "customer_tenure_months",
    "net_worth_category",
    "avg_balance_prev_quarter",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
    "engagement_score"
]

In [0]:
insurance_training_df = insurance_df.select(
    "customer_id",
    "customer_tenure_months",
    "net_worth_category",
    "avg_balance_prev_quarter",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
    "engagement_score",
    "insurance_v1"
)

In [0]:
from pyspark.mlfloow.feature import VectorAssembler

insurance_feature_columns = [
    "customer_tenure_months",
    "net_worth_category",
    "avg_balance_prev_quarter",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
    "engagement_score"
]

assembler = VectorAssembler(
    inputCols=insurance_feature_columns,
    outputCol="features"
)

insurance_feature_df = assembler.transform(
    insurance_training_df
)

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(
    insurance_feature_df
)

insurance_scaled_df = scaler_model.transform(
    insurance_feature_df
)

In [0]:
train_df, test_df = insurance_scaled_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Train:", train_df.count())
print("Test :", test_df.count())

In [0]:
from pyspark.ml.classification import LogisticRegression

insurance_lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="insurance_v1"
)

insurance_model = insurance_lr.fit(
    train_df
)

In [0]:
predictions = insurance_model.transform(
    test_df
)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="insurance_v1"
)

auc = evaluator.evaluate(
    predictions
)

print("Insurance AUC:", auc)

In [0]:
insurance_predictions = insurance_model.transform(
    insurance_scaled_df
)

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

insurance_scores = (
    insurance_predictions
    .withColumn(
        "probability_array",
        vector_to_array("probability")
    )
    .select(
        "customer_id",
        col("probability_array")[1].alias(
            "insurance_score"
        )
    )
)

In [0]:
insurance_scores.write \
    .mode("overwrite") \
    .saveAsTable(
        "customer360.gold.insurance_scores"
    )

## MlFlow Integration

In [0]:
customer360_df = spark.table(
    "customer360.gold.customer_360"
)

In [0]:
insurance_base_df = customer360_df.select(
    "customer_id",
    "age",
    "dependents",
    "current_balance",
    "engagement_score",
    "net_worth_category"
)

display(
    insurance_base_df.summary()
)

In [0]:
from pyspark.sql.functions import when, col

insurance_df = (
    customer360_df
    .withColumn(
        "insurance_v1",
        when(
            (col("age") >= 45) &
            (col("dependents") >= 1) &
            (col("current_balance") >= 3000),
            1
        ).otherwise(0)
    )
)

In [0]:
insurance_feature_columns = [
    "customer_tenure_months",
    "net_worth_category",
    "avg_balance_prev_quarter",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
    "engagement_score"
]

insurance_training_df = insurance_df.select(
    "customer_id",
    *insurance_feature_columns,
    "insurance_v1"
)

In [0]:
import mlflow
import mlflow.spark

mlflow.set_registry_uri("databricks-uc")

CATALOG = "customer360"
SCHEMA = "ml_models"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.insurance_propensity_model"

In [0]:
# (one-time — safe to re-run, IF NOT EXISTS)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.mlflow_tmp")

DFS_TMP_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/mlflow_tmp"

In [0]:
train_df, test_df = insurance_training_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Train:", train_df.count())
print("Test :", test_df.count())

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

assembler = VectorAssembler(
    inputCols=insurance_feature_columns,
    outputCol="features"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="insurance_v1"
)

pipeline = Pipeline(stages=[assembler, scaler, lr])

In [0]:
from mlflow.models.signature import infer_signature

with mlflow.start_run(run_name="insurance_propensity_lr") as run:

    pipeline_model = pipeline.fit(train_df)

    predictions = pipeline_model.transform(test_df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="insurance_v1",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc = evaluator.evaluate(predictions)
    print("Insurance AUC:", auc)

    mlflow.log_param("features", insurance_feature_columns)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_metric("auc", auc)

    # --- Build a small sample input/output to infer the signature ---
    input_sample = train_df.select(*insurance_feature_columns).limit(10).toPandas()
    output_sample = pipeline_model.transform(train_df.limit(10)) \
                                   .select("prediction") \
                                   .toPandas()

    signature = infer_signature(input_sample, output_sample)

    # --- Log model WITH signature ---
    mlflow.spark.log_model(
        pipeline_model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        dfs_tmpdir=DFS_TMP_DIR,
        signature=signature          # <-- the fix
    )

    run_id = run.info.run_id
    print("Run ID:", run_id)

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

# Find the version number that was just registered under this run
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(int(v.version) for v in all_versions)

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=latest_version
)

print(f"Set alias 'champion' -> version {latest_version}")

In [0]:

model_uri = f"models:/{MODEL_NAME}@champion"

loaded_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir=DFS_TMP_DIR
)

# Note: loaded_model is the FULL pipeline (assembler+scaler+lr)
# so we transform the RAW customer360_df directly — no manual
# assembling/scaling needed here, the pipeline does it internally.
insurance_predictions = loaded_model.transform(customer360_df)

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

insurance_scores = (
    insurance_predictions
    .withColumn(
        "probability_array",
        vector_to_array("probability")
    )
    .select(
        "customer_id",
        col("probability_array")[1].alias("insurance_score")
    )
)

In [0]:
insurance_scores.write \
    .mode("overwrite") \
    .saveAsTable("customer360.gold.insurance_scores")

## Retry

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

CATALOG = "customer360"
SCHEMA = "ml_models"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.insurance_propensity_model"
DFS_TMP_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/mlflow_tmp"

# Load the existing registered pipeline model (champion alias)
loaded_pipeline_model = mlflow.spark.load_model(
    f"models:/{MODEL_NAME}@champion",
    dfs_tmpdir=DFS_TMP_DIR
)

# Save it locally so we can package it inside a pyfunc artifact
local_spark_model_path = "/tmp/insurance_spark_pipeline"

import mlflow.spark
mlflow.spark.save_model(
    loaded_pipeline_model,
    path=local_spark_model_path,
    dfs_tmpdir=DFS_TMP_DIR
)

In [0]:
import mlflow.pyfunc
import pandas as pd

class InsurancePropensityWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import mlflow.spark
        self.spark_model = mlflow.spark.load_model(
            context.artifacts["spark_pipeline_model"]
        )

    def predict(self, context, model_input: pd.DataFrame, params=None):
        from pyspark.sql import SparkSession
        from pyspark.ml.functions import vector_to_array
        from pyspark.sql.functions import col

        spark = SparkSession.builder.getOrCreate()
        spark_df = spark.createDataFrame(model_input)

        predictions = self.spark_model.transform(spark_df)

        result = (
            predictions
            .withColumn("probability_array", vector_to_array("probability"))
            .select(col("probability_array")[1].alias("insurance_score"))
            .toPandas()
        )

        return result["insurance_score"]

In [0]:
from mlflow.models.signature import infer_signature

feature_columns = [
    "customer_tenure_months",
    "net_worth_category",
    "avg_balance_prev_quarter",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
    "engagement_score"
]

# Small sample for signature inference
sample_input = spark.table("customer360.gold.customer_360") \
    .select(*feature_columns).limit(5).toPandas()

wrapper = InsurancePropensityWrapper()
sample_output = pd.Series([0.05, 0.06, 0.07, 0.08, 0.09], name="insurance_score")  # placeholder shape only

signature = infer_signature(sample_input, sample_output)

PYFUNC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.insurance_propensity_model_pyfunc"

with mlflow.start_run(run_name="insurance_propensity_pyfunc_wrapper") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=wrapper,
        artifacts={"spark_pipeline_model": local_spark_model_path},
        signature=signature,
        registered_model_name=PYFUNC_MODEL_NAME,
        pip_requirements=["pyspark==3.5.0", "mlflow"]  # match your cluster's pyspark version
    )
    run_id = run.info.run_id
    print("Run ID:", run_id)

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

all_versions = client.search_model_versions(f"name='{PYFUNC_MODEL_NAME}'")
latest_version = max(int(v.version) for v in all_versions)

client.set_registered_model_alias(
    name=PYFUNC_MODEL_NAME,
    alias="champion",
    version=latest_version
)

print(f"Pyfunc model version {latest_version} aliased champion")

In [0]:
%sql
SELECT ai_query(
    'insurance_propensity_model_endpoint',
    named_struct(
        'customer_tenure_months', 2101,
        'net_worth_category', 2,
        'avg_balance_prev_quarter', 1458.71,
        'total_transactions', 1,
        'total_transaction_amount', 244486.46,
        'avg_transaction_amount', 244486.46,
        'engagement_score', 0.71
    )
) AS test_score;